# 08 — Valores y Vectores Propios (Eigenvalues & Eigenvectors)

**Objetivo:** Entender los eigenvectores como las 'direcciones naturales' de una transformación lineal, con aplicaciones en análisis de covarianza y ranking.

## Contexto matemático

Un **vector propio** $\mathbf{v}$ y **valor propio** $\lambda$ de $A$ satisfacen:

$$A\mathbf{v} = \lambda\mathbf{v}$$

$A$ solo escala $\mathbf{v}$ (por $\lambda$), no cambia su dirección.

Para encontrar $\lambda$, resolvemos el **polinomio característico**:

$$\det(A - \lambda I) = 0$$

Para cada $\lambda_i$, el eigenvector $\mathbf{v}_i$ se obtiene resolviendo $(A-\lambda_i I)\mathbf{v}_i=\mathbf{0}$.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Cálculo de eigenvalores y eigenvectores

In [ ]:
# Matriz de covarianza simple 2×2
Sigma = np.array([[3.0, 1.8],
                  [1.8, 2.0]])

# np.linalg.eig para matrices generales (eigenvalues pueden ser complejos)
eigenvalues, eigenvectors = np.linalg.eig(Sigma)

# np.linalg.eigh para matrices simétricas (eigenvalues siempre reales, más estable)
eigenvalues_h, eigenvectors_h = np.linalg.eigh(Sigma)

print('Eigenvalores (eig) :', eigenvalues.round(4))
print('Eigenvalores (eigh):', eigenvalues_h.round(4), ' ← eigh ordena de menor a mayor')

print('\nEigenvectores (columnas):')
print(eigenvectors_h.round(4))

# Verificación Av = λv
for i in range(2):
    lam = eigenvalues_h[i]
    v = eigenvectors_h[:, i]
    print(f'\nλ{i+1}={lam:.4f}: Av={Sigma@v.round(4)}, λv={lam*v.round(4)}, iguales: {np.allclose(Sigma@v, lam*v)}')

## 2 — Eigendescomposición (Diagonalización)

Si $A \in \mathbb{R}^{n\times n}$ tiene $n$ eigenvectores linealmente independientes:

$$A = P D P^{-1}$$

donde $P$ = matriz de eigenvectores (columnas) y $D$ = matriz diagonal de eigenvalores.

**Para matrices simétricas** (Teorema Espectral): $P$ es **ortogonal** ($P^T = P^{-1}$):

$$A = P D P^T$$

**Potencias:** $A^k = P D^k P^{-1}$, donde $D^k$ es simplemente elevar cada $\lambda_i$ a la $k$.

In [ ]:
# Eigendescomposición de Sigma
P = eigenvectors_h
D = np.diag(eigenvalues_h)

# Reconstrucción A = P D P^T (Sigma es simétrica)
Sigma_rec = P @ D @ P.T
print('Reconstrucción P D P^T:')
print(Sigma_rec.round(6))
print('¿Coincide con Sigma original?', np.allclose(Sigma, Sigma_rec))
print('\nP es ortogonal (P^T P = I)?', np.allclose(P.T @ P, np.eye(2)))

# Potencias via diagonalización
k = 5
Sigma_k_diag = P @ np.diag(eigenvalues_h**k) @ P.T
Sigma_k_direct = np.linalg.matrix_power(Sigma, k)
print(f'\nSigma^{k} (diagonalización):')
print(Sigma_k_diag.round(2))
print(f'¿Coincide con matrix_power?', np.allclose(Sigma_k_diag, Sigma_k_direct))

## 3 — Covarianza de portfolio: varianza explicada por dirección

Cuando descomponemos la **matriz de covarianza** de métricas de portfolio, los eigenvectores dan las **direcciones de máxima varianza** (componentes principales). Los eigenvalores dicen cuánta varianza hay en cada dirección.

In [ ]:
# 5 métricas de portfolio fintech: [Retorno, Volatilidad, Drawdown, Sharpe, Beta]
np.random.seed(42)
n_obs = 150
data = np.random.multivariate_normal(
    mean=np.zeros(5),
    cov=[
        [ 1.0,  0.7, -0.6,  0.8, 0.3],
        [ 0.7,  1.0, -0.8,  0.5, 0.6],
        [-0.6, -0.8,  1.0, -0.5,-0.4],
        [ 0.8,  0.5, -0.5,  1.0, 0.2],
        [ 0.3,  0.6, -0.4,  0.2, 1.0],
    ],
    size=n_obs
)
Cov = np.cov(data.T)
lambdas, vecs = np.linalg.eigh(Cov)

# eigh da eigenvalores de menor a mayor; invertir para convención (mayor primero)
lambdas = lambdas[::-1]
vecs    = vecs[:, ::-1]

varianza_explicada = lambdas / lambdas.sum() * 100
varianza_acum = np.cumsum(varianza_explicada)

nombres = ['Retorno', 'Volatilidad', 'Drawdown', 'Sharpe', 'Beta']
print(f'{"Componente":<15} {"λ":>8} {"Var %":>8} {"Acum %":>8}')
print('-'*45)
for i in range(5):
    print(f'PC{i+1:<11}    {lambdas[i]:>8.3f} {varianza_explicada[i]:>7.1f}% {varianza_acum[i]:>7.1f}%')

## 4 — Visualización: eigenvectores como ejes naturales de la transformación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: eigenvectores de Sigma 2×2 con elipse de confianza
ax = axes[0]
# Datos simulados con la covarianza Sigma
np.random.seed(42)
datos2d = np.random.multivariate_normal([0, 0], Sigma, size=300)
ax.scatter(datos2d[:, 0], datos2d[:, 1], alpha=0.25, s=15, color='#adb5bd')

# Eigenvectores escalados por sqrt(eigenvalor)
lams_h, vecs_h = np.linalg.eigh(Sigma)
colores_ev = ['#4361ee', '#f72585']
for i in range(2):
    escala = np.sqrt(lams_h[i]) * 2
    v = vecs_h[:, i] * escala
    ax.annotate('', xy=v, xytext=-v,
                arrowprops=dict(arrowstyle='<->', color=colores_ev[i], lw=2.5))
    ax.text(v[0]*1.1, v[1]*1.1, f'v{i+1} (λ={lams_h[i]:.2f})',
            color=colores_ev[i], fontsize=10, fontweight='bold')
ax.set_aspect('equal')
ax.set_title('Eigenvectores como ejes naturales\nde la distribución (×√λ)')
ax.axhline(0, color='#ccc', lw=0.8); ax.axvline(0, color='#ccc', lw=0.8)

# Panel 2: varianza explicada del portfolio
ax2 = axes[1]
ax2.bar(range(1, 6), varianza_explicada, color='#4361ee', alpha=0.8, label='Var por PC')
ax2_r = ax2.twinx()
ax2_r.plot(range(1, 6), varianza_acum, 'o-', color='#f72585', lw=2, ms=8, label='Acumulado')
ax2_r.axhline(90, color='#adb5bd', linestyle='--', lw=1)
ax2_r.set_ylabel('Varianza acumulada (%)')
ax2.set_xlabel('Componente principal')
ax2.set_ylabel('Varianza explicada (%)')
ax2.set_title('Varianza explicada por eigenvalor\nMatriz de covarianza de portfolio')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

## 5 — Teaser PageRank: eigenvector dominante

PageRank modela la web como un grafo dirigido. La importancia de cada página es el **eigenvector dominante** (eigenvalor $\lambda_1 = 1$) de la matriz de transición.

$$\mathbf{r} = G\mathbf{r} \implies G\mathbf{r} = 1 \cdot \mathbf{r}$$

En analytics: podemos rankear canales o landing pages por influencia en la red de conversión.

In [ ]:
# Red de 5 canales: qué canal 'referencia' a otro (tráfico cruzado)
G_raw = np.array([
    [0, 0.3, 0.4, 0.2, 0.1],  # Organic envía tráfico a...
    [0.2, 0, 0.5, 0.1, 0.2],
    [0.1, 0.3, 0, 0.4, 0.2],
    [0.3, 0.2, 0.1, 0, 0.4],
    [0.4, 0.1, 0.2, 0.3, 0],
])
# Normalizar columnas → matriz estocástica de columnas
G = G_raw / G_raw.sum(axis=0)

lambdas_pr, vecs_pr = np.linalg.eig(G)
# El eigenvalor dominante es 1; el eigenvector correspondiente es el PageRank
idx_dom = np.argmax(np.abs(lambdas_pr.real))
pagerank = np.abs(vecs_pr[:, idx_dom].real)
pagerank /= pagerank.sum()   # normalizar a suma=1

ch_names = ['Organic', 'Paid', 'Email', 'Referral', 'Direct']
orden = np.argsort(pagerank)[::-1]
print(f'Eigenvalor dominante: {lambdas_pr[idx_dom].real:.4f}')
print('\nPageRank de canales (influencia en la red):')
for i in orden:
    bar = '█' * int(pagerank[i]*50)
    print(f'  {ch_names[i]:<10}: {pagerank[i]:.4f}  {bar}')

## Resumen

| Concepto | Fórmula | NumPy |
|---|---|---|
| Eigenvalor / Eigenvector | $A\mathbf{v}=\lambda\mathbf{v}$ | `np.linalg.eig(A)` |
| Matrices simétricas | Eigenvalores reales garantizados | `np.linalg.eigh(A)` |
| Diagonalización | $A = PDP^{-1}$ | Manual con eig |
| Teorema Espectral | $A=PDP^T$ si $A=A^T$ | `eigh` → $P$ ortogonal |
| Potencias | $A^k = PD^kP^{-1}$ | `diag**k` |
| Varianza explicada | $\lambda_i / \sum\lambda_j$ | Con `eigh` en covarianza |
| PageRank | Eigenvector dominante ($\lambda=1$) | `eig` en matriz estocástica |

---

**¡Curso completo!** Resumen de los 8 módulos:

| Módulo | Concepto central |
|---|---|
| 01 | Vectores: operaciones, normas, producto punto |
| 02 | Matrices: tipos especiales, traspuesta, covarianza |
| 03 | Sistemas lineales: Gauss, solve, 3 casos |
| 04 | Multiplicación matricial: regla, propiedades, Markov |
| 05 | Determinantes: área, singularidad, multicolinealidad |
| 06 | Inversas: Gauss-Jordan, solve vs inv, pseudoinversa |
| 07 | Espacios vectoriales: rango, base, espacio nulo |
| 08 | Eigenvalores: descomposición espectral, PCA, PageRank |